# Hyperparameter Tunning (Red Wine)

In [1]:
# Imports
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath(".."))

from preprocessing import get_X_y
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from sklearn.metrics import f1_score , precision_score , recall_score
from scipy.stats import uniform , randint
from sklearn.ensemble import RandomForestClassifier

In [2]:
# Data Load

df = pd.read_csv(r"../dataset/winequality-red.csv" , sep = ";")
df

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
...,...,...,...,...,...,...,...,...,...,...,...,...
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,5


In [3]:
# Preprocessing 

X , y = get_X_y(df)

X

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4
...,...,...,...,...,...,...,...,...,...,...,...
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2


In [4]:
# Split and Weight

X_train , X_test , y_train , y_test = train_test_split(
    X ,
    y ,
    test_size = 0.2 ,
    stratify = y , 
    random_state = 98
)

weight = compute_sample_weight(
    y = y_train ,
    class_weight = "balanced"
)

In [52]:
# 1st Try

model = XGBClassifier(
    n_jobs = 1 ,
    subsample = 0.8 ,
    colsample_bytree = 0.8 ,
)

param = {
    "n_estimators" : randint(200 , 401) ,
    "max_depth" : randint(4 , 9) ,
    "min_child_weight" : [1 , 2 , 3 , 4 , 5],
    "learning_rate" : uniform(0.01 , 0.29) ,
    "gamma" : uniform(0 , 5) ,
    "reg_lambda" : uniform(0 , 30) ,
    "reg_alpha" : uniform(0 , 10)
}

srch = RandomizedSearchCV(
    estimator = model ,
    param_distributions = param ,
    cv = 3 ,
    n_iter = 50 ,
    scoring="f1_weighted",
    n_jobs = -1
)

srch.fit(
    X_train ,
    y_train ,
)

model = srch.best_estimator_

print(srch.best_params_)
print()
y_pred = model.predict(X_test)
print(f"F1 Score : {f1_score(y_test , y_pred , average = "weighted")}")
print(f"Precision : {precision_score(y_test , y_pred , average = "weighted")}")
print(f"Recall Score : {recall_score(y_test , y_pred , average = "weighted")}")

{'gamma': np.float64(0.0452244653445516), 'learning_rate': np.float64(0.1733111668032613), 'max_depth': 7, 'min_child_weight': 1, 'n_estimators': 262, 'reg_alpha': np.float64(1.648017511207801), 'reg_lambda': np.float64(25.855039970229114)}

F1 Score : 0.7094981679767851
Precision : 0.7010227980737043
Recall Score : 0.721875


C:\Users\sailj\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [78]:
# 1st Try

model = RandomForestClassifier()

model.fit(
    X_train ,
    y_train
)

y_pred = model.predict(X_test)

print(print(model.get_params()))
print()
print(f"F1 Score : {f1_score(y_test , y_pred , average = "weighted")}")
print(f"Precision : {precision_score(y_test , y_pred , average = "weighted")}")
print(f"Recall Score : {recall_score(y_test , y_pred , average = "weighted")}")

{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': None, 'verbose': 0, 'warm_start': False}
None

F1 Score : 0.7402576660604642
Precision : 0.7227050631797289
Recall Score : 0.759375


C:\Users\sailj\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


# Cleary Xgboost is not performing well and i also tried to train parameter for random forest but this dataset works well in raw model